# Capacitated assignment workflow

This notebook demonstrates a small end-to-end workflow:

1. Generate synthetic historical orders for several sectors and a table of cycle windows.
2. Forecast the shape and total volume of a future cycle.
3. Assign forecast requests to service days with a simple greedy capacity heuristic.
4. Validate the forecast and assignment against a held-out cycle with tables and graphs.


#### Import libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

## Problem setup

A request is an aggregate number of orders from one sector on a requested date. The assignment heuristic may move part of that request to another date in the same cycle when the preferred date is full. The objective is intentionally simple: preserve capacity feasibility while keeping movement away from the requested date small.

This is a teaching example, so the data is intentionally small and the heuristic is transparent rather than production-ready.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

SEED = 42
rng = np.random.default_rng(SEED)
SECTORS = ["A", "B", "C", "D"]
CYCLE_LENGTH = 14
LAST_HISTORICAL_CYCLE = 6
VALIDATION_CYCLE = 7
N_CYCLES = VALIDATION_CYCLE

## Part 1 — Synthetic orders and cycle windows

The two core input tables are:

- `orders`: one row per sector and day, representing the raw historical order feed.
- `cycles`: one row per cycle, defining when the cycle opens and closes.

The synthetic orders combine a sector baseline, a repeating within-cycle pattern, a gentle cycle trend, and random noise.

In [ ]:
cycle_starts = pd.date_range("2025-01-01", periods=N_CYCLES, freq=f"{CYCLE_LENGTH}D")
cycles = pd.DataFrame(
    {
        "cycle_id": np.arange(1, N_CYCLES + 1),
        "open_date": cycle_starts,
    }
)
cycles["close_date"] = cycles["open_date"] + pd.Timedelta(days=CYCLE_LENGTH - 1)
cycles["n_days"] = (cycles["close_date"] - cycles["open_date"]).dt.days + 1

display(cycles)

In [ ]:
sector_baseline = {"A": 34, "B": 43, "C": 29, "D": 51}
sector_factor = {"A": 1.00, "B": 1.08, "C": 0.92, "D": 1.12}
cycle_factor = np.array([0.91, 0.96, 1.00, 1.04, 1.08, 1.12, 1.16])
within_cycle_pattern = np.array(
    [0.72, 0.86, 0.97, 1.08, 1.18, 1.25, 1.10, 0.90, 0.78, 0.84, 1.00, 1.12, 1.20, 0.98]
)
within_cycle_pattern = within_cycle_pattern / within_cycle_pattern.mean()

order_rows = []
for cycle_id, cycle_start in enumerate(cycle_starts, start=1):
    for day_in_cycle in range(CYCLE_LENGTH):
        order_date = cycle_start + pd.Timedelta(days=day_in_cycle)
        for sector in SECTORS:
            expected_orders = (
                sector_baseline[sector]
                * sector_factor[sector]
                * cycle_factor[cycle_id - 1]
                * within_cycle_pattern[day_in_cycle]
            )
            orders = int(rng.poisson(expected_orders))
            order_rows.append(
                {
                    "order_date": order_date,
                    "cycle_id": cycle_id,
                    "day_in_cycle": day_in_cycle + 1,
                    "sector": sector,
                    "orders": orders,
                }
            )

orders = pd.DataFrame(order_rows)
historical_orders = orders[orders["cycle_id"] <= LAST_HISTORICAL_CYCLE].copy()
validation_orders = orders[orders["cycle_id"] == VALIDATION_CYCLE].copy()

print(f"Generated {len(orders):,} sector-day observations.")
print(
    f"Forecast training rows: {len(historical_orders):,}; held-out validation rows: {len(validation_orders):,}."
)
display(orders.head(12))

## Part 2 — Forecast the cycle shape and time series

We use a deliberately simple two-stage forecast:

1. **Cycle shape:** for each sector, calculate each historical day's share of that sector's cycle total, then average those shares across historical cycles.
2. **Cycle volume:** fit a straight-line trend to each sector's historical cycle totals and extrapolate one cycle ahead.

The forecast for a sector/day is `forecast cycle total × average day share`.

In [ ]:
historical_cycle_totals = (
    historical_orders.groupby(["cycle_id", "sector"], as_index=False)["orders"]
    .sum()
    .rename(columns={"orders": "cycle_total"})
)
shape_observations = historical_orders.merge(historical_cycle_totals, on=["cycle_id", "sector"])
shape_observations["order_share"] = shape_observations["orders"] / shape_observations["cycle_total"]
cycle_shape = shape_observations.groupby(["sector", "day_in_cycle"], as_index=False)[
    "order_share"
].mean()


def extrapolate_cycle_total(cycle_totals):
    """Extrapolate one positive cycle total from a short linear trend."""
    values = cycle_totals.to_numpy(dtype=float)
    x_values = np.arange(len(values), dtype=float)
    slope = np.polyfit(x_values, values, deg=1)[0] if len(values) > 1 else 0.0
    return max(0.0, values[-1] + slope)


forecast_cycle_totals = (
    historical_cycle_totals.sort_values("cycle_id")
    .groupby("sector")["cycle_total"]
    .apply(extrapolate_cycle_total)
    .rename("forecast_cycle_total")
    .reset_index()
)

target_cycle = cycles.loc[cycles["cycle_id"].eq(VALIDATION_CYCLE)].iloc[0]
forecast_orders = cycle_shape.merge(forecast_cycle_totals, on="sector")
forecast_orders["cycle_id"] = VALIDATION_CYCLE
forecast_orders["requested_date"] = target_cycle["open_date"] + pd.to_timedelta(
    forecast_orders["day_in_cycle"] - 1, unit="D"
)
forecast_orders["forecast_orders"] = np.rint(
    forecast_orders["order_share"] * forecast_orders["forecast_cycle_total"]
).astype(int)

print("Forecast cycle totals by sector")
display(forecast_cycle_totals)
print("Average historical cycle shape")
display(cycle_shape.pivot(index="day_in_cycle", columns="sector", values="order_share"))

In [ ]:
forecast_daily = forecast_orders.groupby("requested_date", as_index=False)["forecast_orders"].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for sector in SECTORS:
    sector_shape = cycle_shape[cycle_shape["sector"].eq(sector)]
    axes[0].plot(
        sector_shape["day_in_cycle"],
        sector_shape["order_share"],
        marker="o",
        label=f"Sector {sector}",
    )
axes[0].set_title("Forecasted cycle shape")
axes[0].set_xlabel("Day in cycle")
axes[0].set_ylabel("Share of sector cycle volume")
axes[0].legend()

axes[1].plot(
    forecast_daily["requested_date"],
    forecast_daily["forecast_orders"],
    marker="o",
    color="tab:blue",
    label="Forecast requests",
)
axes[1].set_title("Forecast time series for validation cycle")
axes[1].set_xlabel("Requested date")
axes[1].set_ylabel("Orders")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()
fig.tight_layout()
plt.show()

## Part 3 — Greedy capacitated assignment

The forecast rows are requests: `(sector, requested_date, quantity)`. Each date in the target cycle has the same service capacity. The heuristic processes requests in date order and assigns each request to its preferred date first, then to the nearest dates with remaining capacity.

The returned allocation can contain multiple rows for one request when a request is split across service dates. A production implementation could replace this heuristic with a MIP, CP-SAT model, or a richer local-search method while keeping the same input/output tables.

In [ ]:
# Set capacity below the busiest forecast day so the assignment has to move some requests.
daily_capacity = max(160, int(np.ceil(forecast_daily["forecast_orders"].max() * 0.84)))
capacity_plan = pd.DataFrame(
    {
        "service_date": pd.date_range(
            target_cycle["open_date"], target_cycle["close_date"], freq="D"
        ),
        "capacity": daily_capacity,
    }
)


def greedy_assign(requests, capacity_plan):
    """Assign requests to nearby dates while respecting daily capacity."""
    remaining_capacity = capacity_plan.set_index("service_date")["capacity"].to_dict()
    allocations = []
    unassigned = []
    request_order = requests.sort_values(
        ["requested_date", "forecast_orders"], ascending=[True, False]
    )

    for request in request_order.itertuples(index=False):
        quantity_left = int(request.forecast_orders)
        candidate_dates = sorted(
            remaining_capacity,
            key=lambda date: (abs((date - request.requested_date).days), date),
        )
        for service_date in candidate_dates:
            assigned = min(quantity_left, remaining_capacity[service_date])
            if assigned == 0:
                continue
            allocations.append(
                {
                    "sector": request.sector,
                    "requested_date": request.requested_date,
                    "service_date": service_date,
                    "assigned_orders": assigned,
                    "displacement_days": abs((service_date - request.requested_date).days),
                }
            )
            remaining_capacity[service_date] -= assigned
            quantity_left -= assigned
            if quantity_left == 0:
                break
        if quantity_left:
            unassigned.append(
                {
                    "sector": request.sector,
                    "requested_date": request.requested_date,
                    "unassigned_orders": quantity_left,
                }
            )

    return pd.DataFrame(allocations), pd.DataFrame(unassigned)


allocations, unassigned = greedy_assign(forecast_orders, capacity_plan)
if unassigned.empty:
    print("All forecast requests were assigned.")
else:
    print("Some forecast requests could not be assigned:")
    display(unassigned)

assigned_by_service_date = allocations.groupby("service_date", as_index=False)[
    "assigned_orders"
].sum()
assignment_summary = capacity_plan.merge(
    assigned_by_service_date, how="left", on="service_date"
).fillna(0)
assignment_summary["remaining_capacity"] = (
    assignment_summary["capacity"] - assignment_summary["assigned_orders"]
)

display(allocations.head(15))
display(assignment_summary)

## Part 4 — Validation

We compare:

- forecast requests with the actual requests in the held-out cycle;
- requests by requested date with assigned work by service date; and
- assigned volume with the daily capacity limit.

The displacement metric measures how far the heuristic moved an order from its requested day.

In [ ]:
actual_daily = (
    validation_orders.groupby("order_date", as_index=False)["orders"]
    .sum()
    .rename(columns={"order_date": "requested_date", "orders": "actual_orders"})
)
daily_validation = (
    forecast_daily.rename(columns={"forecast_orders": "forecast_requests"})
    .merge(actual_daily, how="outer", on="requested_date")
    .merge(
        assigned_by_service_date.rename(
            columns={"service_date": "requested_date", "assigned_orders": "assigned_orders"}
        ),
        how="left",
        on="requested_date",
    )
    .merge(
        capacity_plan.rename(columns={"service_date": "requested_date"}),
        how="left",
        on="requested_date",
    )
    .fillna(0)
    .sort_values("requested_date")
)
daily_validation["forecast_error"] = (
    daily_validation["forecast_requests"] - daily_validation["actual_orders"]
)
daily_validation["utilization"] = daily_validation["assigned_orders"] / daily_validation["capacity"]

total_actual = daily_validation["actual_orders"].sum()
total_forecast = daily_validation["forecast_requests"].sum()
mae = daily_validation["forecast_error"].abs().mean()
wmape = daily_validation["forecast_error"].abs().sum() / total_actual
total_displacement = (allocations["assigned_orders"] * allocations["displacement_days"]).sum()
weighted_displacement = total_displacement / allocations["assigned_orders"].sum()

metrics = pd.Series(
    {
        "forecast_orders": total_forecast,
        "actual_orders": total_actual,
        "daily_MAE": mae,
        "daily_WMAPE": wmape,
        "assigned_orders": allocations["assigned_orders"].sum(),
        "weighted_displacement_days": weighted_displacement,
        "maximum_utilization": daily_validation["utilization"].max(),
    },
)
display(daily_validation)
display(metrics.to_frame("value"))

In [ ]:
forecast_by_sector = forecast_orders.pivot(
    index="requested_date", columns="sector", values="forecast_orders"
).fillna(0)
actual_by_sector = validation_orders.pivot(
    index="order_date", columns="sector", values="orders"
).fillna(0)
assigned_by_sector_date = allocations.pivot_table(
    index="service_date", columns="sector", values="assigned_orders", aggfunc="sum"
).fillna(0)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(
    daily_validation["requested_date"],
    daily_validation["forecast_requests"],
    marker="o",
    label="Forecast requests",
)
axes[0, 0].plot(
    daily_validation["requested_date"],
    daily_validation["actual_orders"],
    marker="o",
    label="Actual requests",
)
axes[0, 0].set_title("Requests by day: forecast vs actual")
axes[0, 0].set_ylabel("Orders")
axes[0, 0].legend()

axes[0, 1].bar(
    daily_validation["requested_date"],
    daily_validation["forecast_requests"],
    alpha=0.65,
    label="Requested",
)
axes[0, 1].plot(
    daily_validation["requested_date"],
    daily_validation["assigned_orders"],
    color="tab:orange",
    marker="o",
    label="Assigned on service date",
)
axes[0, 1].plot(
    daily_validation["requested_date"],
    daily_validation["capacity"],
    color="black",
    linestyle="--",
    label="Capacity",
)
axes[0, 1].set_title("Greedy assignment and capacity")
axes[0, 1].set_ylabel("Orders")
axes[0, 1].legend()

forecast_by_sector.plot(kind="bar", stacked=True, ax=axes[1, 0], width=0.85)
axes[1, 0].set_title("Forecast requests by sector and day")
axes[1, 0].set_xlabel("Requested date")
axes[1, 0].set_ylabel("Orders")
axes[1, 0].tick_params(axis="x", rotation=45)

assigned_by_sector_date.plot(kind="bar", stacked=True, ax=axes[1, 1], width=0.85)
axes[1, 1].axhline(daily_capacity, color="black", linestyle="--", label="Capacity")
axes[1, 1].set_title("Assigned orders by service day")
axes[1, 1].set_xlabel("Service date")
axes[1, 1].set_ylabel("Orders")
axes[1, 1].tick_params(axis="x", rotation=45)
axes[1, 1].legend()

fig.suptitle("Validation cycle: requests, assignment, and capacity", fontsize=15)
fig.tight_layout()
plt.show()

## Interpretation and next steps

The first plot checks demand forecasting, while the second checks whether the assignment respects capacity. Because capacity is deliberately tighter than the peak forecast, some requests should be moved to nearby dates; the weighted displacement reports the amount of that movement.

For a more realistic study, the next extensions would be: request-level records instead of aggregates, sector-specific eligibility and service costs, multiple resources or shifts, holidays and missing-data treatment, forecast intervals, and a comparison of this greedy solution with an exact optimization model.